<a href="https://colab.research.google.com/github/benoitmialet/artificial_neural_networks_labs/blob/main/ann_lab_04_cancer_tuning_regularization_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Regularization with `PyTorch` and Hyperparameter Tuning with `Optuna` (Breast Cancer dataset)

The binary classifier we built in a previous lab showed signs of overfitting.

We are going to take it as an example to learn:
* Regularization to avoid or limit overfitting
* Hyperparameter tuning to improve model training

# Baseline training method

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.metrics import roc_curve, auc, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter

from tqdm import tqdm

RANDOM_STATE = 101
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

## Data processing

Let's use our previous data pipeline to prepare data. We will take only the first 5000th samples to accelerate processes.

In [ ]:
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target
X = df.drop('target', axis=1).values
y = df['target'].values

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size = 0.2, random_state = RANDOM_STATE)

scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_val = scaler.transform(X_val)

class CancerDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = CancerDataset(X_train, y_train)
val_dataset = CancerDataset(X_val, y_val)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

print(len(train_dataset[0][0]))

## Model class, evaluation method

In [ ]:
class BCNet(nn.Module):
    def __init__(self, input_dim, hidden1=30, hidden2=15):
        super().__init__()
        self.ff = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Linear(hidden2, 1)
        )
    def forward(self, x):
        return self.ff(x)

model = BCNet(input_dim=30)

def evaluate_classification(model, device, loader, criterion):
    model.eval()
    model = model.to(device)
    losses, probs, trues = [], [], []

    with torch.inference_mode():
        for x_batch, y_batch in loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            logits = model(x_batch)
            loss = criterion(logits, y_batch)
            losses.append(loss.item() * len(x_batch))

            batch_probs = torch.sigmoid(logits).squeeze(1).cpu().numpy() # we move from potential GPU tensor to CPU numpy array
            probs.extend(batch_probs)
            trues.extend(y_batch.squeeze(1).cpu().numpy())

    avg_loss = np.sum(losses) / len(loader.dataset)
    probs = np.array(probs)
    preds = (probs > 0.5).astype(int)
    trues = np.array(trues)

    return avg_loss, probs, preds, trues

## Training loop

We are going to put our traning loop in a method, that we can call everytime we need to customize our model training. It will avoid to copy paste the code too much and to be lost in out experiments.

In [ ]:
def model_train_and_log(model, train_loader, val_loader, epochs, device, criterion, optimizer, writer):
  for epoch in tqdm(range(1, epochs)):
      model.train()
      model = model.to(device)
      running_loss = 0
      for x_batch, y_batch in train_loader:
          x_batch, y_batch = x_batch.to(device), y_batch.to(device)
          optimizer.zero_grad()
          logits = model(x_batch)
          loss = criterion(logits, y_batch) * len(x_batch)
          loss.backward()
          optimizer.step()
          running_loss += loss.item()

      avg_loss = running_loss / len(train_loader.dataset)
      val_loss, val_probs, val_preds, val_trues = evaluate_classification(model, device, val_loader, criterion)

      writer.add_scalar("Loss/train", avg_loss, epoch)
      writer.add_scalar("Loss/val", val_loss, epoch)
      writer.add_scalar("Accuracy/val", accuracy_score(val_trues, val_preds), epoch)
  writer.close()


## Training: Baseline model

Let's first use our refactored code to make a baseline training

In [ ]:
device

In [ ]:
writer = SummaryWriter("./runs/breast_cancer_bc")
model = BCNet(input_dim=30)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model_train_and_log(model, train_loader, val_loader, 300, device, criterion, optimizer, writer)

# Tensorboard monitoring

In [ ]:
# On Jupyter Notebbook or vscode:
# tensorboard --logdir=runs  # then go to --> http://localhost:6006

# on Google colab:
%load_ext tensorboard
%tensorboard --logdir "./runs"

# Regularization methods

## Training: Baseline model + Batch Normalization

In [ ]:
class BCNetBN(nn.Module):
    def __init__(self, input_dim, hidden1=30, hidden2=15):
        super().__init__()
        self.ff = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.BatchNorm1d(hidden1), # new
            nn.ReLU(),
            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2), # new
            nn.ReLU(),
            nn.Linear(hidden2, 1)
        )
    def forward(self, x):
        return self.ff(x)

In [ ]:
writer = SummaryWriter("./runs/breast_cancer_bc_bn")
model = BCNetBN(input_dim=30)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model_train_and_log(model, train_loader, val_loader, 200, device, criterion, optimizer, writer)

What do we see there?
* `BCNet` is performing worse whith batch normalization
* it can be expected, as BN works well with deep and complex networks, when covariances shift occurs. As a regularization process, it will slightly increase bias. On a rather shallow neural network, there is no covariiance shift and we will observe only this bias increase without taking the benefits of BN.

## Training: Baseline model + Droupout

Dropout is a regularization technique where, during training, a random proportion of neurons are temporarily deactivated (their outputs are set to zero). Their weights are not updated for that training step. This prevents the network from relying too heavily on specific neurons, and forces it to learn more robust, distributed representations.

We are going to define the model in a customizable way, so that we could tune hypermarameters later.
* number of hidden layers (except the first one)
* number of neurons per layer
* proportion of connexions to dropout during training (regularization)

In [ ]:
class BCNetDropout(nn.Module):
  def __init__(self, input_dim, hidden1=30, hidden2=15, dropout=0.2):
      super().__init__()
      self.ff = nn.Sequential(
          nn.Linear(input_dim, hidden1),
          nn.ReLU(),
          nn.Dropout(dropout),
          nn.Linear(hidden1, hidden2),
          nn.ReLU(),
          nn.Dropout(dropout),
          nn.Linear(hidden2, 1)
      )
  def forward(self, x):
      return self.ff(x)


In [ ]:
writer = SummaryWriter("./runs/breast_cancer_bc_dropout")
model = BCNetDropout(input_dim=30)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model_train_and_log(model, train_loader, val_loader, 200, device, criterion, optimizer, writer)

## Training: Baseline model + L2 Regularization

L2 weight decay is a regularization technique where an extra penalty is added to the loss function, proportional to the square of the weights. This gently pushes the model’s weights toward smaller values, preventing them from growing too large, and forcing network to rely on all weights. As a result, the model becomes less sensitive to noise, and generalizes better.

In [ ]:
writer = SummaryWriter("./runs/breast_cancer_bc_l2")
# Add L2 regularization through weight_decay
model = BCNet(input_dim=30)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

model_train_and_log(model, train_loader, val_loader, 200, device, criterion, optimizer, writer)

## Training: Baseline model + L2 + Droupout

In [ ]:
writer = SummaryWriter("./runs/breast_cancer_bc_l2_dropout")
model = BCNetDropout(input_dim=30)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

model_train_and_log(model, train_loader, val_loader, 200, device, criterion, optimizer, writer)

## What do we see there?

* Regularization with Dropout lowers the loss on validation curve as compared to validation curve with baseline training
* L2 Regularization lowers the loss on validation curve, even more than Dropout Does
* BUT, combining Dropout + L2 seems to "ruin" everything: validation curve is higher than validation curve during baseline training.

## Possible explanations: what do you think?

- Dropout and L2 each reduce overfitting and improve generalization when used separately.
- When combined, the regularization becomes too strong, leading to underfitting and higher validation loss.
- The chosen dropout rate or weight decay value may be too aggressive when applied together.

**Conclusion:**
- Regularization can help, but its effects depend on dataset size, model complexity, and hyperparameter tuning.
- Combining methods is not always beneficial — too much regularization can hurt performance.
- Regularization is not universally helpful. It must be applied when there is evidence of overfitting, and tuned carefully, or it can make a model worse by over-regularizing.

# Hyperparameter tuning with `Optuna`

https://optuna.readthedocs.io/en/stable/index.html

`Optuna` is a python library that helps to automatize hyperparameter tuning.
It uses a bayesian optimisation algorythm, that will avoid to try out every possible value for parameters.

In [ ]:
!pip install optuna

Let's define a function just to train the model, without evaluation and logging

In [ ]:
def model_train(model, train_loader, val_loader, epochs, device, criterion, optimizer):
  for epoch in tqdm(range(1, epochs)):
      model.train()
      model = model.to(device)
      running_loss = 0
      for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model(x_batch)
        loss = criterion(logits, y_batch) * len(x_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

      avg_loss = running_loss / len(train_loader.dataset)
      val_loss, val_probs, val_preds, val_trues = evaluate_classification(model, device, val_loader, criterion)
  return val_loss

In [ ]:
import optuna
# optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    # n_neurons = trial.suggest_int("n_neurons_h1", 16, 128)
    lr = trial.suggest_float("learning_rate", 1e-4, 1e-3)
    dropout = trial.suggest_float("dropout", 0.0, 0.4)
    l2 = trial.suggest_float("l2", 0.0, 1e-3)
    model = BCNetDropout(
        input_dim=30,
        hidden1=30,
        hidden2=16,
        dropout=dropout
    ).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=l2)
    avg_val_loss = model_train(model, train_loader, val_loader, 100, device, criterion, optimizer)

    return avg_val_loss

# Create a study with a persistent storage
study = optuna.create_study()
study.optimize(objective, n_trials=50)

print(f"Best value: {study.best_value} (params: {study.best_params})")

In [ ]:
optuna.visualization.plot_optimization_history(study)

In [ ]:
optuna.visualization.plot_slice(study)

In [ ]:
optuna.visualization.plot_contour(study, params=["dropout", "l2"])

In [ ]:
optuna.visualization.plot_contour(study, params=["learning_rate", "l2"])

In [ ]:
study.best_params

## Now let's finally train the model again with best parameters and compare it to the baseline training!

In [ ]:
writer = SummaryWriter("./runs/breast_cancer_bc_tuned")
model = BCNetDropout(
  input_dim=30,
  hidden1=30,
  hidden2=15,
  dropout=0
)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(
  model.parameters(),
  lr=study.best_params["learning_rate"],
  weight_decay=study.best_params["l2"]
)

model_train_and_log(model, train_loader, val_loader, 200, device, criterion, optimizer, writer)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, accuracy_score, confusion_matrix


test_loss, test_probs, _, test_trues = evaluate_classification(model, device, val_loader, criterion) # again, we use validation data only for lab convenience.

# Binary predictions
test_preds = (test_probs > 0.5).astype(int)

# Accuracy
accuracy = accuracy_score(test_trues, test_preds)
print("Test Accuracy:", accuracy)

# Confusion Matrix
cm = confusion_matrix(test_trues, test_preds, normalize='true')
# plt.figure(figsize=(9, 9))
sns.heatmap(cm, annot=True, fmt=".3f", linewidths=.5, square=True, cmap='Blues')
plt.ylabel('Actual label')
plt.xlabel('Predicted label')
plt.title(f"Accuracy = {accuracy:.3f}", size=15)
plt.show()

# ROC Curve
fpr, tpr, _ = roc_curve(test_trues, test_probs)
plt.plot(fpr, tpr, label=f"AUC={auc(fpr,tpr):.2f}")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()



## Save & Load the model

In [ ]:
# Save and load model
torch.save(model.state_dict(), 'regression_model_tuned.pth')

# Load model example
loaded_model = BCNetDropout(input_dim=30).to(device)
loaded_model.load_state_dict(torch.load('regression_model_tuned.pth'))